## **Model for Deployment**

In [1]:
import pandas as pd
import numpy as np
import joblib
import time
import warnings
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb

warnings.filterwarnings('ignore')
SEED = 42

print("🔄 Loading dataset...")
df = pd.read_csv('Heart.csv', na_values=['NA', '?', ''])
df['AHD'] = df['AHD'].map({'No': 0, 'Yes': 1})
X = df.drop(columns=['AHD', 'HD'])
y = df['AHD']

num_features = ['Age', 'RestBP', 'Chol', 'MaxHR', 'Oldpeak', 'Ca']
cat_features = ['Sex', 'ChestPain', 'Fbs', 'RestECG', 'ExAng', 'Slope', 'Thal']

print("⚙️ Configuring preprocessor...")
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, num_features),
    ('cat', categorical_transformer, cat_features)
], remainder='drop')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=SEED, stratify=y)
pos_weight_train = (y_train == 0).sum() / (y_train == 1).sum()

print("🏗️ Building pipelines...")
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('rf', RandomForestClassifier(random_state=SEED, n_jobs=-1, class_weight='balanced'))
])
xgb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('xgb', xgb.XGBClassifier(random_state=SEED, eval_metric='logloss', n_jobs=-1, scale_pos_weight=pos_weight_train))
])

param_dist_rf = {
    'rf__n_estimators': [100, 200, 300],
    'rf__max_depth': [None, 10, 15],
    'rf__min_samples_split': [2, 5],
    'rf__class_weight': ['balanced', 'balanced_subsample']
}
param_dist_xgb = {
    'xgb__n_estimators': [100, 200, 300],
    'xgb__max_depth': [3, 4, 5],
    'xgb__learning_rate': [0.05, 0.1, 0.2],
    'xgb__subsample': [0.8, 1.0]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

print("🚀 Training Random Forest (this may take a minute)...")
rf_search = RandomizedSearchCV(rf_pipeline, param_dist_rf, n_iter=15, cv=cv, scoring='roc_auc', n_jobs=-1, random_state=SEED)
rf_search.fit(X_train, y_train)

print("🚀 Training XGBoost...")
xgb_search = RandomizedSearchCV(xgb_pipeline, param_dist_xgb, n_iter=15, cv=cv, scoring='roc_auc', n_jobs=-1, random_state=SEED)
xgb_search.fit(X_train, y_train)

print("💾 Saving pre-trained models to disk...")
joblib.dump(rf_search.best_estimator_, 'rf_model.pkl')
joblib.dump(xgb_search.best_estimator_, 'xgb_model.pkl')
joblib.dump(preprocessor, 'preprocessor.pkl')

print("✅ Success! Models saved as 'rf_model.pkl', 'xgb_model.pkl', and 'preprocessor.pkl'")

🔄 Loading dataset...
⚙️ Configuring preprocessor...
🏗️ Building pipelines...
🚀 Training Random Forest (this may take a minute)...
🚀 Training XGBoost...
💾 Saving pre-trained models to disk...
✅ Success! Models saved as 'rf_model.pkl', 'xgb_model.pkl', and 'preprocessor.pkl'
